# 🕵️ Dia 59 — Forense de rede com Wireshark e tshark

**Investigador Cyber 90 Dias** · Fase 4 · Rank Jutsu `B`

> Baseado no *Day 59: Network forensics with Wireshark and tshark* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-4-digital-forensics-incident-investigation/day-59.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

A Serra Azul não guarda pacotes, só proxy e firewall. Rafa espalhou na mesa os horários das conexões do `SA-CX-07` para `atualiza-modulo.example`: *"'Uns cinco minutos' não vai para o relatório. Média, mínimo, máximo, desvio. E me explica aquele buraco das 12h07."*

Neste notebook você vai:

1. **Gerar um pcap sintético** (só cabeçalhos, nenhum conteúdo real) e lê-lo byte a byte com `struct`, como o tshark faz.
2. **Resumir conversas e I/O** (o equivalente a `-z conv,tcp` e `-z io,stat`).
3. **Medir o *beacon*** com média, mediana, desvio e coeficiente de variação.
4. **Montar as listas de arestas do Gephi** para o LAB-P4 e para a agência Centro (CSV e GEXF).
5. **Jogar o quiz "filtro de captura ou de exibição?"**.

🐍 Só biblioteca padrão. Para a Parte A do original (a sua captura real), há uma célula opcional com `tshark`.

## 1. Carregando os logs de firewall

In [ ]:
# ------------------------------------------------------------
# CARREGADOR OFFLINE-FIRST
# Tenta baixar cada arquivo do GitHub (raw). Se a rede falhar,
# usa a cópia EMBUTIDA aqui embaixo (comprimida para caber no notebook).
# Nos dois casos, calculamos o SHA-256: é a nossa "cadeia de custódia" de bolso.
# ------------------------------------------------------------
import base64          # para decodificar as cópias embutidas
import hashlib         # SHA-256 de cada arquivo carregado
import urllib.request  # download simples, sem bibliotecas externas
import zlib            # as cópias embutidas estão comprimidas com zlib

# Endereço "cru" dos dados do LAB-P4 no repositório original (ThiefDroppers)
RAW_LAB = ("https://raw.githubusercontent.com/thiefdroppers/cyber-investigator-skills/main/"
           "curriculum/phase-4-digital-forensics-incident-investigation/resources/case-lab-p4/")
# Endereço "cru" dos dados sintéticos da Operação Isca neste repositório
RAW_BR = ("https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/"
          "caso-continuo/dados/")

# Plano B: cópias fiéis dos arquivos, feitas a partir do repositório local.
EMBUTIDOS = {
    "lab:logs/firewall.log": """
eNqt291qXDcQB/D7PMW+wC6ab+lACsXpdU1JH8AhDgkYu2Q3hLx9tc42tY4d/7XxgMlV7N9KGs3M0VlxYd8W2ZJsyJcSSymbD18L
bX6/uPjj8u3m7cXlhsqu/2jZWSzaWuXN9rcNNd6VHe+4LKqy+fzl5vr11c3N3dftx8Phn/327sth8+7b4Xr/Ws21veIBovZzyPq/
vFgpOkAEIaJmvoK0QKi1Jg8hmYBUaQWZ4qlz8gFyBEVVlzVUn4WcFjOqA8QQEtE6OLGwIEdFw4Ylqsjh2lqsoYBLVCvTMHMEoRJq
K6jPNxqRxhjdOBZCi60hZTSivo1iGJFCiIN1cOpC2PEqet52leiLtIYUxlxtZYgFVjigHttryGAsKEkbgpscQ/3DDVBbJOCIgric
F3SVYpUXWt8kMLqjthimLuAa0ZhR+6oSTN3dqXUYUMPbtVpZQYx3UfgI4SVS6yl1BU3sIqZVLMCZi2J1HBF15Qf05q8/L78zre6M
OlV20WPBezq5h+517h9gEantu/T++sPVl5vD9v317bcT448MftbwHvpk9kID79LWdFwZgStDEraGHFdv8jGRMs471WQFPZfgTkXV
Y6xBgvNOUaprKJ5dnurHjH2qQb+6PAyMvjzMp3j+VUNgX9XLlQ9pjWEWUG1jWuuQEgwBqirn5U/XXujXkMF0Y1zKeSFgscqffe4J
1+waet7mYVFex7TBzaPVx1oqBjvSvneG1pcZFe1jRSh9jc6D3IJtBSmGqseYqHHpIeayhhoekfGQ4NjwGgmtoWMw8COIi/S/SSS7
HpTauNTVXmV+CO33H7fvrvaHT3e3P3IpPyGRQok4RWLBkqdIgiXWFKnXMyRJSZB6JcJjkkiRKKCkliJxw1JLkQSvk3GKZISljCjv
jT2WXFMkMizVFInxOnlLkRTPXuSMyQqWMsakM1WjRoo0kSOap0gC10mKpEgKZ0+IUiSbkDJmzyaqhjClSDhHSEol7L+lWIoUSRxK
KimSNixljMknqoZoTZEIx57ljInxOllLkXDVEM+ZPcOzFxnVPfB5Wpif98herY4H+ico8JA8ZUiMlyml5MbEo4ZUSZEUp72U4n58
XoZSy3gAqAsVLNUUiWF50pQ2oi4Co1yppEj44VNTGpY20VgqtRQJFw1lTpEYr1NKw9ImGksVTZFw0VB5cYaV+1cWOPa0pEj42Ehf
3oTdSzIxpkiRFMeeZcweTbR7ajVFItiEqVOKhA8JNHJmbyIiImf28COh1oydyxNdmDZKkRiPqbUUSWB1t6IpEj7MMcqYPZmoufby
g/l7iSakSJFwzTXWFEkalixFMtiXm2SMSfsPllqKhHO5pVRCXQTvJ/MUSXGOSKlP2hsxLGXEnk10y+Y1RcK53MJTJHzgaxEpkuLZ
q5wiTeSImrFzvWNQapIiMd65raZI+IzFC6dIuC//77sRL5UMZlinjKoRHcNSpEj4pZCn1NyYeHHsIikSfgLwlCfqmOgsPeWJ+skv
cD+SIkfC66QtRSK8TkYpEu5h3SxFwi8b3HLWCZ8becopwcxJoltLkXBv5K4pEn4h6Sn9Xpt4IekhKdJENqqUI+EckdEbUZnJRs1T
JJyNokiKJHA/RcYpwfH6k2MpY0wzZ5bBJUWiCSlSJPxGMjJOCbqEz8JCOUXC759CM8Y08+XHUEuR8FlYZPQRNHM6GiYpEq4aYTmz
N5EjMqo7yUQPG15SJPwF6XBPkfCbrghNkSZyRM2IPYVPn1aIM7JReyrDPvibSsMqSTne1noMfbo9XH++vbr5cbGhri6GycM7vT+7
S8Xt0RvJcy7SyBP3W/5+czl8dJPhPXhXjk38g9G8v92vR9P4CYZ/MmvfGWl2Oob4f4SMb21VJmllPXP2/My10v9HvOgKktx/5xHc
b7F66ienb4OICbdX/wLJa96W""",
    "br:dia-57/fw-serraazul.log": """
eNqtmdtqHEcQhu/9FPsCu9S5uwYcCCL3vvALyKAQgxKHSEb47V2zjhJ1zbKlaQv2Slrpmzr9dRgCsiPCEfQAbQGIz+H3p+PD7eHX
m5vfPnw8fLz5cEA4EZwEToqLKroejr8c0OkUPz8hLiJ8+Ofr/d372/v7L0/HPx4f/344fvn6ePj07fHu4b00g/6ORpLF5xqJFhUw
f0lir0gKBpZICAvSNRIv2tlsILWK1AgcMkkK79Ei3tD3kRRFs/cofM7XSLqoSzzMS5JVJOPwQybFf7pqU1tJNNhEUnqvk3km+UJS
xIm740CiikTNuSUSS00K5402QUmi+EIm9YVa4T1mkKGeuCJxY8REkr4AFjaZYtsXJwKRHKfQAKk0oiPyvsolB8/eiywWLzQC0Ics
lzJOJj3lXrhOF5CCZM6DRlBZT82wtUSK6JIWGdFgrNyowcomR7RM6ou0QiO4oe3T8mYdKJM8Er2wCQH7D1KPDDkhrL/EpUOGvWT5
BZJgSSJaSQQcNkUinlqpseJukFihsaAli3EvqwnjFqVUo2w3itrGKi664RklsNuD0aU2iRGJCyUMn5P9v8TAuojRXTLMCljkO7kM
Wku1C5WkZ5JHf6jNogmzumbBiGZ1XQQjdwy1D3JbCkYodGpWZ5JaaRbifrOIeo4WV5Uc/cpBRnWqFZeUL4BaaRVN5CCB5uqSeGq9
Hixx9Z1WIUrLkiERLKzN0hmzSDMsOmSvgkVOQ2mVY4yRWE4L8XrcDLN8wizeyK6GD72EMc3AMIvGeVAuR4w2+BDL8ZYjxFkx1F/R
TZCnfAhjeXmMTYuUXTLyd0Y0iDJs3eYqGLcJGGN0L4TMC+2oPakyIfSNNcFie2SttkcT2rf/mBFsQVKnvbaZrpzm3YC1coaPnRh0
nyIKcOuZFGlfTqFoOCX0nmD0Kul4PirsS3vMaR+TDVe3Eupt5w7OrJLNYliAa7P8Tao5YEzFZMO9DQs/l6AOlnOQ8eJk+P946+v8
3exHHZ/xemHhf7r79Oft5/vj57+eaws3tRUo1AoVje9tUJf28AElTd8ERRdXoeRA+nc0fD2qd9kke8yu9dqFbUqdNGsuW3016a0P
2/i6VxeVZeJZneLxoFanPtFKom1hhr1KnRxn+pZlH2pYVg7YNLPkEUvfwgiLJS9WNt43icYcuun9+po8jAqYCtgwReF6dF+Qa9jU
yNYhw7hYX3WxGIZk3/GYYiXfkDS6cmkW8s8vyuuRPz51HqLPCMeY9CtJF6oDRvzzl42AhZJSqR3EM5MNjofkFYYLt+o43o13vsQg
aTlgpPVxPMyaGjbGy+EK64VZeL7ZDGZhebVmZ8tJv441tXDIzGgIZhmmxQx1humM0jtn2PpHUMNmKqxzzsOyh60vCJVo3zmPQTmX
l3ph1vrqBNpwYSOvTTKFd98BEId8CA==""",
}

REGISTRO_DE_CARGA = []   # (arquivo, origem, sha256): vai para as suas notas de método


def carregar(chave):
    """chave = 'lab:logs/firewall.log' ou 'br:dia-57/proxy-sa.log'. Devolve o texto."""
    prefixo, rel = chave.split(":", 1)                    # separa de onde vem e o caminho
    url = (RAW_LAB if prefixo == "lab" else RAW_BR) + rel  # monta a URL raw
    reserva = zlib.decompress(base64.b64decode(EMBUTIDOS[chave].replace("\n", "")))  # bytes da cópia embutida
    try:
        with urllib.request.urlopen(url, timeout=10) as r:  # tenta a rede, com tempo limite
            dados, origem = r.read(), "rede"
    except Exception:                                       # sem rede, 404, timeout: tanto faz
        dados, origem = reserva, "embutido"
    if origem == "rede" and dados != reserva:              # a versão online mudou desde que copiamos?
        print(f"⚠️  {rel}: versão online difere da embutida; usando a embutida (números do dia batem com ela)")
        dados, origem = reserva, "embutido (online divergente)"
    sha = hashlib.sha256(dados).hexdigest()                # impressão digital do que vamos analisar
    REGISTRO_DE_CARGA.append((rel, origem, sha))           # anota para o relatório
    return dados.decode("utf-8")


print(f"{len(EMBUTIDOS)} arquivos disponíveis. Use carregar('...').")

In [ ]:
FW_LAB = carregar("lab:logs/firewall.log").splitlines()      # LAB-P4, hora de Nova York
FW_SA = carregar("br:dia-57/fw-serraazul.log").splitlines()  # Serra Azul, hora de Brasília
print(len(FW_LAB), "linhas LAB-P4 |", len(FW_SA), "linhas Serra Azul")

In [ ]:
# Tabela de carga: o que foi lido, de onde e com qual hash.
for rel, origem, sha in REGISTRO_DE_CARGA:     # uma linha por arquivo
    print(f"{rel:40s} {origem:10s} {sha[:16]}…")  # 16 primeiros caracteres bastam para conferir de olho

## 2. Um pcap sintético, escrito e lido com `struct`

O formato **pcap clássico** é simples: um cabeçalho global de 24 bytes e, para cada pacote, um cabeçalho de 16 bytes (segundos, microssegundos, tamanho capturado, tamanho original) seguido dos bytes do pacote. Vamos gerar pacotes **TCP SYN** (só cabeçalhos Ethernet/IP/TCP, sem dados) reproduzindo as conexões do `SA-CX-07` que o firewall registrou.

Nada aqui é tráfego real: os horários e IPs vêm do log sintético, e não há conteúdo nenhum.

In [ ]:
import struct                                              # empacota números em bytes (e desempacota)
import socket                                              # converte "10.20.40.57" em 4 bytes
from datetime import datetime, timedelta, timezone

UTC = timezone.utc
try:
    from zoneinfo import ZoneInfo
    SP, NY = ZoneInfo("America/Sao_Paulo"), ZoneInfo("America/New_York")
except Exception:                                          # sem tzdata: fusos fixos (válidos nas datas do caso)
    SP, NY = timezone(timedelta(hours=-3)), timezone(timedelta(hours=-4))


def parse_fw(linha, fuso):
    """'2026-10-05 08:19:38 fw-sa ACCEPT TCP 10.20.40.57:51018 -> 198.51.100.201:80 rule=... bytes=792'"""
    p = linha.split()
    dt = datetime.fromisoformat(f"{p[0]} {p[1]}").replace(tzinfo=fuso).astimezone(UTC)   # sempre UTC
    src, sport = p[5].rsplit(":", 1)
    dst, dport = p[7].rsplit(":", 1)
    return {"dt": dt, "acao": p[3], "proto": p[4], "src": src, "sport": int(sport),
            "dst": dst, "dport": int(dport), "bytes": int(p[9].split("=")[1])}


fluxos_sa = [parse_fw(l, SP) for l in FW_SA]


def pacote_syn(src, dst, sport, dport):
    """Ethernet (14) + IPv4 (20) + TCP (20) = 54 bytes, flag SYN. Checksums zerados (é sintético)."""
    eth = b"\x02\x00\x00\x00\x00\x01" + b"\x02\x00\x00\x00\x00\x02" + b"\x08\x00"  # MACs locais + tipo IPv4
    ip = struct.pack("!BBHHHBBH4s4s", 0x45, 0, 40, 0, 0x4000, 64, 6, 0,
                     socket.inet_aton(src), socket.inet_aton(dst))           # versão/IHL, TTL 64, proto 6 = TCP
    tcp = struct.pack("!HHIIBBHHH", sport, dport, 0, 0, 5 << 4, 0x02, 64240, 0, 0)  # 0x02 = SYN
    return eth + ip + tcp


def escrever_pcap(caminho, fluxos):
    with open(caminho, "wb") as f:
        f.write(struct.pack("<IHHiIII", 0xA1B2C3D4, 2, 4, 0, 0, 65535, 1))  # magic, versão 2.4, linktype 1 = Ethernet
        for fl in fluxos:
            pkt = pacote_syn(fl["src"], fl["dst"], fl["sport"], fl["dport"])
            ts = fl["dt"].timestamp()
            f.write(struct.pack("<IIII", int(ts), 0, len(pkt), len(pkt)))   # cabeçalho do pacote
            f.write(pkt)


escrever_pcap("sa-cx-07-sintetico.pcap", [f for f in fluxos_sa if f["proto"] == "TCP"])
import os
print("sa-cx-07-sintetico.pcap:", os.path.getsize("sa-cx-07-sintetico.pcap"), "bytes")

In [ ]:
def ler_pcap(caminho):
    """O que o tshark faz na primeira camada: cabeçalho global, depois pacote a pacote."""
    pacotes = []
    with open(caminho, "rb") as f:
        magic, vmaj, vmin, _, _, snap, link = struct.unpack("<IHHiIII", f.read(24))
        assert magic == 0xA1B2C3D4 and link == 1, "não é pcap Ethernet little-endian"
        while True:
            cab = f.read(16)
            if len(cab) < 16:                                   # fim do arquivo
                break
            seg, usec, incl, orig = struct.unpack("<IIII", cab)
            dados = f.read(incl)
            ihl = (dados[14] & 0x0F) * 4                         # tamanho do cabeçalho IP
            src, dst = socket.inet_ntoa(dados[26:30]), socket.inet_ntoa(dados[30:34])
            sport, dport = struct.unpack("!HH", dados[14 + ihl:14 + ihl + 4])
            flags = dados[14 + ihl + 13]
            pacotes.append({"t": seg + usec / 1e6, "src": src, "dst": dst, "sport": sport,
                            "dport": dport, "syn": bool(flags & 0x02), "ack": bool(flags & 0x10)})
    return pacotes


pkts = ler_pcap("sa-cx-07-sintetico.pcap")
print(len(pkts), "pacotes lidos")

# Filtro de EXIBIÇÃO, à moda Python: ip.addr == 198.51.100.140 && tcp.flags.syn == 1 && tcp.flags.ack == 0
filtro = lambda p: "198.51.100.140" in (p["src"], p["dst"]) and p["syn"] and not p["ack"]
visiveis = [p for p in pkts if filtro(p)]
print(len(visiveis), "pacotes passam no filtro de exibição (os outros continuam no arquivo)")

## 3. `-z conv,tcp` e `-z io,stat` em Python

In [ ]:
from collections import Counter, defaultdict

# Conversas: par (origem, destino) -> conexões e bytes (do firewall, porque o pcap sintético só tem SYN)
conv = defaultdict(lambda: [0, 0])
for f in fluxos_sa:
    conv[(f["src"], f["dst"])][0] += 1
    conv[(f["src"], f["dst"])][1] += f["bytes"]
print(f"{'Origem':16s} {'Destino':16s} {'Con.':>5s} {'Bytes':>10s}")
for (s, d), (n, b) in sorted(conv.items(), key=lambda kv: -kv[1][1])[:6]:
    print(f"{s:16s} {d:16s} {n:5d} {b:10d}")

# I/O graph em texto: conexões do SA-CX-07 por intervalo de 10 minutos
print("\nI/O (conexões de 10.20.40.57 por 10 min, UTC)")
io = Counter(f["dt"].replace(minute=f["dt"].minute // 10 * 10, second=0) for f in fluxos_sa if f["src"] == "10.20.40.57")
for faixa in sorted(io):
    print(f"{faixa:%H:%M}  {'█' * io[faixa]} {io[faixa]}")

## 4. O *beacon*: média, mediana, desvio, coeficiente de variação

Mude o formulário para outros destinos e veja a diferença entre um programa e uma pessoa.

In [ ]:
#@title Destino a medir { display-mode: "form" }
origem = "10.20.40.57"  #@param {type:"string"}
destino = "198.51.100.140"  #@param {type:"string"}

import statistics as st

conexoes = sorted((f for f in fluxos_sa if f["src"] == origem and f["dst"] == destino), key=lambda f: f["dt"])
gaps = [(b["dt"] - a["dt"]).total_seconds() for a, b in zip(conexoes, conexoes[1:])]
if len(gaps) >= 2:
    media, mediana, desvio = st.mean(gaps), st.median(gaps), st.pstdev(gaps)
    print(f"{len(conexoes)} conexões, {len(gaps)} intervalos")
    print(f"média {media:.1f} s | mediana {mediana:.1f} s | mín {min(gaps):.0f} | máx {max(gaps):.0f} | desvio {desvio:.1f} | CV {desvio/media:.3f}")
    # Outliers: conexões com volume muito acima da mediana de bytes
    med_b = st.median(c["bytes"] for c in conexoes)
    fora = [c for c in conexoes if c["bytes"] > 10 * med_b]
    for c in fora:
        print(f"⚠️  outlier: {c['dt']:%H:%M:%SZ} {c['bytes']:,} bytes (mediana {med_b:,.0f})".replace(",", "."))
    no_ritmo = [c for c in conexoes if c not in fora]
    g2 = [(b["dt"] - a["dt"]).total_seconds() for a, b in zip(no_ritmo, no_ritmo[1:])]
    print(f"sem o outlier: média {st.mean(g2):.1f} s | desvio {st.pstdev(g2):.1f} s | CV {st.pstdev(g2)/st.mean(g2):.3f}")
else:
    print("Poucas conexões para medir ritmo. Tente 10.20.40.57 → 198.51.100.140.")

> 💡 Repare: com o envio de 3 MB no meio, aparecem dois intervalos de ~150 s e o desvio dispara. Tirando o *outlier*, o ritmo volta a ~300 s com CV baixíssimo. **Média sozinha esconde as duas coisas.** E "regular" ainda não é "malicioso": atualizadores legítimos também batem a cada 5 minutos.

In [ ]:
# Comparação: os acessos HTTPS de rotina de um caixa vizinho (pessoa navegando)
vizinho = sorted((f for f in fluxos_sa if f["src"] == "10.20.40.53"), key=lambda f: f["dt"])
gv = [(b["dt"] - a["dt"]).total_seconds() for a, b in zip(vizinho, vizinho[1:])]
print(f"10.20.40.53 (vários destinos): média {st.mean(gv):.0f} s, desvio {st.pstdev(gv):.0f} s, CV {st.pstdev(gv)/st.mean(gv):.2f}")

## 5. Listas de arestas para o Gephi (passo 6 do original)

Mesma lógica do `awk`: somar bytes por (origem, destino, ação). Conferimos as 3 primeiras linhas do LAB-P4 com as do original.

In [ ]:
def arestas(fluxos):
    agg = defaultdict(lambda: [0, 0])
    for f in fluxos:
        k = (f["src"], f["dst"], f["acao"])
        agg[k][0] += f["bytes"]
        agg[k][1] += 1
    return sorted(((s, d, a, b, n) for (s, d, a), (b, n) in agg.items()), key=lambda r: -r[3])

def salvar_csv(linhas, caminho):
    with open(caminho, "w", encoding="utf-8") as f:
        f.write("Source,Target,Action,Weight,Connections\n")
        for r in linhas:
            f.write(",".join(map(str, r)) + "\n")

fluxos_lab = [parse_fw(l, NY) for l in FW_LAB]
a_lab, a_sa = arestas(fluxos_lab), arestas(fluxos_sa)
salvar_csv(a_lab, "fw-edges.csv")
salvar_csv(a_sa, "fw-edges-serraazul.csv")

ESPERADO = [("10.10.30.17", "198.51.100.23", "ACCEPT", 48213904, 1),
            ("203.0.113.45", "10.10.20.5", "ACCEPT", 465560, 113),
            ("10.10.40.61", "192.0.2.29", "ACCEPT", 85848, 1)]
print("LAB-P4 bate com o original?", "✅" if a_lab[:3] == ESPERADO else "❌")
for r in a_lab[:3]:
    print("  ", ",".join(map(str, r)))
print("\nSerra Azul (top 4):")
for r in a_sa[:4]:
    print("  ", ",".join(map(str, r)))

In [ ]:
# GEXF: o formato nativo do Gephi, com rótulos de host já prontos (File > Open).
from xml.sax.saxutils import escape
ROTULOS = {"10.10.20.5": "bastion01", "10.10.30.17": "fs01", "10.10.40.57": "WS-FIN-07",
           "10.20.40.57": "SA-CX-07", "10.20.5.20": "webmail-sa", "203.0.113.77": "serraazul-pix.example",
           "198.51.100.201": "lnk.example", "198.51.100.140": "atualiza-modulo.example"}

def gexf(linhas, caminho):
    nos = sorted({r[0] for r in linhas} | {r[1] for r in linhas})
    with open(caminho, "w", encoding="utf-8") as f:
        f.write('<?xml version="1.0" encoding="UTF-8"?>\n<gexf xmlns="http://gexf.net/1.3" version="1.3">\n')
        f.write('<graph defaultedgetype="directed"><nodes>\n')
        for n in nos:
            f.write(f'<node id="{n}" label="{escape(ROTULOS.get(n, n))}"/>\n')
        f.write("</nodes><edges>\n")
        for i, (s, d, a, b, c) in enumerate(linhas):
            f.write(f'<edge id="{i}" source="{s}" target="{d}" weight="{b}" label="{a} {c} con."/>\n')
        f.write("</edges></graph></gexf>\n")

gexf(a_lab, "lab-p4-flows.gexf"); gexf(a_sa, "serraazul-flows.gexf")
print("💾 fw-edges.csv, fw-edges-serraazul.csv, lab-p4-flows.gexf, serraazul-flows.gexf")

## 6. 🎮 Quiz: filtro de captura ou de exibição?

Checkpoint BR: pelo menos 5 de 6.

In [ ]:
# ------------------------------------------------------------
# QUIZ: função genérica de múltipla escolha
# MODO_JOGO = True  -> pergunta e conta acertos
# MODO_JOGO = False -> só mostra gabarito e explicação
# ------------------------------------------------------------
def jogar_quiz(perguntas, modo_jogo=True, minimo=None):
    acertos = 0                                            # placar
    for n, (enunciado, opcoes, certa, porque) in enumerate(perguntas, start=1):
        print(f"\n[{n}/{len(perguntas)}] {enunciado}")    # o enunciado
        for letra, texto in opcoes.items():                # as alternativas a), b), c)
            print(f"   {letra}) {texto}")
        if modo_jogo:
            resposta = ""
            for _ in range(5):                             # até 5 tentativas de digitar uma letra válida
                resposta = input("   Sua resposta: ").strip().lower()
                if resposta in opcoes:
                    break
            ok = (resposta == certa)                       # acertou?
            acertos += ok
            print("   ✅ Acertou!" if ok else f"   ❌ Era {certa}).")
        else:
            print(f"   Gabarito: {certa})")
        print(f"   Por quê: {porque}")
    if modo_jogo:
        print(f"\n🏁 Resultado: {acertos}/{len(perguntas)}")
        if minimo is not None:
            print("Checkpoint BR cumprido. 🥷" if acertos >= minimo
                  else f"Checkpoint BR pede pelo menos {minimo}. Releia o README e tente de novo.")
    return acertos

In [ ]:
MODO_JOGO = True

PERGUNTAS = [
    ("`host 198.51.100.140 and tcp port 443` é um filtro de:",
     {"a": "exibição", "b": "captura (BPF): o que ele exclui nunca é gravado", "c": "Suricata"},
     "b", "Sintaxe BPF, usada no momento da gravação."),
    ("Você gravou só `tcp port 443` e depois quis ver o DNS que veio antes. Dá?",
     {"a": "sim, é só mudar o filtro de exibição", "b": "não: o filtro de captura descartou o DNS", "c": "sim, com --export-objects"},
     "b", "Por isso em investigação se captura amplo e filtra na exibição."),
    ("Intervalos de 292 a 309 s, com dois de ~150 s no meio. A melhor descrição é:",
     {"a": "beacon de malware confirmado", "b": "conexões regulares (~300 s, CV baixo) e uma conexão extra fora do ritmo",
      "c": "navegação humana"},
     "b", "Regularidade é comportamento; o extra é o envio de 3 MB, descrito à parte."),
    ("Sobre os 3.145.728 bytes enviados às 12:07:10Z, sem pcap, você pode afirmar:",
     {"a": "quantos bytes saíram, de onde, para onde e quando", "b": "que era a planilha de cooperados", "c": "que eram senhas"},
     "a", "Fluxo não mostra conteúdo."),
    ("`dns.flags.rcode == 3` mostra:",
     {"a": "consultas DNS por TXT", "b": "respostas NXDOMAIN", "c": "DNS sobre HTTPS"},
     "b", "rcode 3 = NXDOMAIN, comum em malware que gera domínios."),
    ("Por que olhar a dispersão e não só a média dos intervalos?",
     {"a": "porque beacons reais têm jitter e um outlier distorce a média",
      "b": "porque a média é sempre zero", "c": "porque o Gephi exige"},
     "a", "É o item 'sem notas' do checkpoint do original."),
]
jogar_quiz(PERGUNTAS, MODO_JOGO, minimo=5)

## 7. (Opcional) Parte A do original, na sua VM

Capture **só o tráfego da sua própria VM de laboratório**. As linhas `!` são puladas no teste e não rodam no Colab sem `tshark`.

In [ ]:
!sudo apt-get -y install tshark
!tshark -r sa-cx-07-sintetico.pcap -q -z conv,tcp
!tshark -r sa-cx-07-sintetico.pcap -Y 'ip.addr == 198.51.100.140' -T fields -e frame.time_epoch | head
print("Dica: o pcap sintético abre no Wireshark. Statistics > I/O Graphs mostra o pulso de ~300 s.")

In [ ]:
try:
    from google.colab import files
    for f in ("fw-edges.csv", "fw-edges-serraazul.csv", "lab-p4-flows.gexf", "serraazul-flows.gexf", "sa-cx-07-sintetico.pcap"):
        files.download(f)
except ImportError:
    print("(Fora do Colab: os arquivos ficaram na pasta atual.)")

## ✅ Antes de fechar o Dia 59

- [ ] `fw-edges.csv` do LAB-P4 bate com as 3 linhas do original
- [ ] Ritmo do `SA-CX-07` anotado: n, média, mediana, mín, máx, desvio, CV, e o *outlier* explicado
- [ ] Grafo da agência no Gephi com os destinos do caso rotulados
- [ ] Quiz com pelo menos 5/6

**Amanhã (Dia 60):** Suricata. O ritmo de hoje vira regra, e você testa a regra contra o pcap sintético.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>